# Kaggle GPU Server — Embeddings Endpoint

Sirve `multilingual-e5-large` como endpoint HTTP para que los scripts locales
puedan generar embeddings sin GPU local.

## Prerequisitos
1. GPU activada (Settings → Accelerator → GPU T4 x2)
2. Internet activado (Settings → Internet → On)
3. Secrets configurados en Kaggle Secrets:
   - `EMBEDDING_SERVER_TOKEN` — token de autenticación para el endpoint

## Uso
1. Ejecutar todas las celdas (1-4)
2. Esperar a que la celda 4 muestre la URL de serveo.net
3. Copiar la URL y ponerla en `EMBEDDING_SERVER_URL` en tu `.env` local
4. Ejecutar `scripts/run_discovery.py` o `scripts/run_experimental.py` localmente
5. **NO parar la celda 4** — mantiene la sesión activa y monitoriza el tunnel

In [ ]:
# === 1. Instalar dependencias ===
%pip install -q flask sentence-transformers torch

print("Dependencias instaladas.")

In [ ]:
# === 2. Cargar modelo en GPU ===
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-large"

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cargando {MODEL_NAME} en {device}...")

model = SentenceTransformer(MODEL_NAME, device=device)

# Test rapido
test_vec = model.encode(["test"], normalize_embeddings=True)
print(f"Modelo cargado. Dimensiones: {test_vec.shape[1]}")

In [ ]:
# === 3. Flask server ===
import socket
import threading
from datetime import datetime
from flask import Flask, request, jsonify
from kaggle_secrets import UserSecretsClient

_request_stats = {"count": 0, "last_time": None, "lock": threading.Lock()}

# Comprobar si el puerto ya esta en uso (re-ejecucion de celda)
def port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

if port_in_use(5000):
    print("Puerto 5000 ya esta en uso — el server ya esta corriendo.")
    print("Si necesitas reiniciar, haz Kernel → Restart y re-ejecuta todas las celdas.")
else:
    secrets = UserSecretsClient()
    API_TOKEN = secrets.get_secret("EMBEDDING_SERVER_TOKEN")

    app = Flask(__name__)

    @app.after_request
    def _count_request(response):
        with _request_stats["lock"]:
            _request_stats["count"] += 1
            _request_stats["last_time"] = datetime.now()
        return response

    @app.route("/health", methods=["GET"])
    def health():
        token = request.headers.get("Authorization", "").replace("Bearer ", "")
        if token != API_TOKEN:
            return jsonify({"error": "Unauthorized"}), 401
        return jsonify({
            "status": "ok",
            "model": MODEL_NAME,
            "device": str(model.device),
        })

    @app.route("/embed", methods=["POST"])
    def embed():
        token = request.headers.get("Authorization", "").replace("Bearer ", "")
        if token != API_TOKEN:
            return jsonify({"error": "Unauthorized"}), 401

        data = request.get_json()
        texts = data.get("texts", [])
        if not texts:
            return jsonify({"error": "No texts provided"}), 400

        embeddings = model.encode(texts, normalize_embeddings=True)
        return jsonify({"embeddings": embeddings.tolist()})

    thread = threading.Thread(
        target=lambda: app.run(host="0.0.0.0", port=5000),
        daemon=True,
    )
    thread.start()
    print("Flask server arrancado en puerto 5000")

In [ ]:
# === 4. Tunnel SSH + Keepalive ===
# Usa SSH para crear un tunnel a través de serveo.net (sin límite de datos).
# Si serveo.net no responde, localhost.run como fallback.
# NO parar esta celda mientras se usen los scripts locales.

import re, time, os, subprocess
from datetime import datetime, timedelta
from IPython.display import clear_output

HEARTBEAT_INTERVAL = 30  # seconds
KAGGLE_MAX_HOURS = 12
LOG_PATH = "/tmp/tunnel.log"

_start_time = datetime.now()
_ssh_proc = None


def _get_gpu_mem():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
            text=True,
        ).strip()
        used, total = out.split(", ")
        return f"{used}/{total} MiB"
    except Exception:
        return "N/A"


def _get_tunnel_url():
    try:
        with open(LOG_PATH, "r") as f:
            content = f.read()
        # serveo.net
        match = re.search(r"(https://[a-z0-9]+\.serveo\.net)", content)
        if match:
            return match.group(1)
        # localhost.run fallback
        match = re.search(r"(https://[a-z0-9-]+\.lhr\.life)", content)
        if match:
            return match.group(1)
    except Exception:
        pass
    return None


def _is_tunnel_alive():
    global _ssh_proc
    return _ssh_proc is not None and _ssh_proc.poll() is None


def _start_tunnel():
    global _ssh_proc
    if _ssh_proc and _ssh_proc.poll() is None:
        _ssh_proc.terminate()
        time.sleep(1)
    log_file = open(LOG_PATH, "w")
    _ssh_proc = subprocess.Popen(
        [
            "ssh", "-R", "80:localhost:5000",
            "-o", "StrictHostKeyChecking=no",
            "-o", "ServerAliveInterval=30",
            "-o", "ServerAliveCountMax=3",
            "-o", "ExitOnForwardFailure=yes",
            "serveo.net",
        ],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )


# --- Lanzar tunnel ---
_start_tunnel()

restarts = 0
tunnel_url = None
print("Iniciando tunnel SSH (serveo.net)...")

try:
    while True:
        now = datetime.now()
        elapsed = now - _start_time
        remaining = timedelta(hours=KAGGLE_MAX_HOURS) - elapsed

        # Check tunnel health + get URL
        tunnel_url = _get_tunnel_url()
        if not _is_tunnel_alive():
            restarts += 1
            _start_tunnel()
            # Esperar URL del nuevo tunnel
            deadline = time.time() + 30
            while time.time() < deadline:
                time.sleep(2)
                tunnel_url = _get_tunnel_url()
                if tunnel_url:
                    break
            tunnel_status = f"RESTARTED (x{restarts})" if tunnel_url else f"RESTART FAILED (x{restarts})"
        else:
            tunnel_status = "OK" if tunnel_url else "WAITING..."

        # Request stats
        with _request_stats["lock"]:
            req_count = _request_stats["count"]
            last_req = _request_stats["last_time"]
        last_req_str = last_req.strftime("%H:%M:%S") if last_req else "ninguna"

        # Build status
        elapsed_str = str(elapsed).split(".")[0]
        remaining_str = str(remaining).split(".")[0] if remaining.total_seconds() > 0 else "EXPIRADO"

        clear_output(wait=True)
        if tunnel_url:
            print(f"{'='*50}")
            print(f"EMBEDDING_SERVER_URL={tunnel_url}")
            print(f"{'='*50}")
        print(f"[{now.strftime('%H:%M:%S')}] Heartbeat"
              f" | Elapsed: {elapsed_str}"
              f" | Remaining: {remaining_str}"
              f" | GPU: {_get_gpu_mem()}"
              f" | Requests: {req_count} (last: {last_req_str})"
              f" | Tunnel: {tunnel_status}")

        time.sleep(HEARTBEAT_INTERVAL)

except KeyboardInterrupt:
    print("\nKeepalive detenido manualmente.")